# MacroGAT Gold Optuna -- PTP Continuous Position Sizing

Trains **MacroGATMamba** (3-branch: Spatial + Macro GAT + Technical) with Optuna-searched
hyperparameters. Uses **PredictionToPosition** (4-class quartile head) for continuous
position sizing on 1h forward Gold returns.

Session: 07:00-11:00 (overlap + 1h before US session).

## 1. Environment Setup

In [ ]:
# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print(f"Colab: {IN_COLAB}")

In [ ]:
# Add CTAFlow to path (Colab only)
if IN_COLAB:
    import sys
    sys.path.insert(0, '/content/drive/MyDrive/CTAFlow')

In [ ]:
import gc
import json
import math
import warnings
from collections import defaultdict
from copy import deepcopy
from pathlib import Path
from typing import Dict, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import optuna
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9} GB")

## 2. Configuration

In [ ]:
# --- Ticker ---
TICKER = 'GC'

# --- Architecture ---
USE_AMP = True            # Mixed precision (float16)

# --- Target ---
TARGET_HORIZON_MINUTES = 60   # 1h forward return
BAR_MINUTES = 5               # 5min bars -> target = y_fwd_12

# --- Session Filter ---
SAMPLE_SESSION = "custom"
SAMPLE_SESSION_START = "07:00"  # Overlap + 1h before US session
SAMPLE_SESSION_END = "11:00"

# --- Stride ---
# stride=12 with 5min bars + 60min target -> non-overlapping samples
SAMPLE_STRIDE = 12

# --- Macro ---
MACRO_LOOKBACK = 20       # Trading days of macro history per sample
QUARTILE_WINDOW = 1000    # Rolling window for quartile target bucketing

# --- Selection score weights (matches HybridTCN) ---
SELECTION_SHARPE_WEIGHT = 0.20
SELECTION_SORTINO_WEIGHT = 0.35
SELECTION_PF_WEIGHT = 0.25
SELECTION_RETURN_WEIGHT = 0.20
SELECTION_TOTAL_RETURN_SCALE = 100.0

# --- Paths ---
if IN_COLAB:
    DRIVE_PATH = Path('/content/drive/MyDrive')
    DATA_ROOT = DRIVE_PATH / 'features'
    RESULTS_PATH = DRIVE_PATH / 'results' / 'macro_gat_gc'
else:
    DATA_ROOT = Path('/workspace/model_data')
    RESULTS_PATH = Path('/workspace/results/macro_gat_gc')

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
FRED_KEY_FP = DATA_ROOT / 'fredapi.txt'


print(f"Data root: {DATA_ROOT}")
print(f"Results path: {RESULTS_PATH}")
print(f"Ticker: {TICKER}")
print(f"Target: {TARGET_HORIZON_MINUTES}min forward return")
print(f"Session filter: {SAMPLE_SESSION_START}-{SAMPLE_SESSION_END}")
print(f"Stride: {SAMPLE_STRIDE} bars ({SAMPLE_STRIDE * BAR_MINUTES}min between samples)")
print(f"Macro lookback: {MACRO_LOOKBACK} days")
print(f"Quartile window: {QUARTILE_WINDOW}")
print(f"Selection weights: sharpe={SELECTION_SHARPE_WEIGHT}, sortino={SELECTION_SORTINO_WEIGHT}, pf={SELECTION_PF_WEIGHT}, total_return={SELECTION_RETURN_WEIGHT}")

In [ ]:
# Verify data files
print("Checking data files...")
ticker_dir = DATA_ROOT / TICKER
required = ['intraday.csv', f'{TICKER}_numbars.npz', 'rasterized.npz']
for f in required:
    p = ticker_dir / f
    status = 'OK' if p.exists() else 'MISSING'
    print(f"  {f}: {status}")
    assert p.exists(), f"Missing {p}"

status = "OK" if FRED_KEY_FP.exists() else "MISSING" 
print(f"  fredapi.txt: {status}")

## 3. Load Data via MacroGATContinuousPrep

In [ ]:
from CTAFlow.data.datasets.macro_gat import (
    MacroGATContinuousPrep,
    MacroGATDataset,
    macro_gat_collate_fn,
    unpack_macro_gat_batch,
    build_macro_gat_loaders,
)
from CTAFlow.models.prep.intraday_continuous import SessionSpec
from CTAFlow.features.macro_gat_prep import NODE_ORDER

    
print("Loading data...")

with open(FRED_KEY_FP, 'r') as f:
    api_key = f.read()

prep = MacroGATContinuousPrep(
    sessions=[SessionSpec(SAMPLE_SESSION, SAMPLE_SESSION_START, SAMPLE_SESSION_END)],
    bar_minutes=BAR_MINUTES,
    target_horizon_minutes=TARGET_HORIZON_MINUTES,
    macro_lookback=MACRO_LOOKBACK,
    quartile_window=QUARTILE_WINDOW,
)
prep.load(root_dir=DATA_ROOT, ticker=TICKER, fred_api_key=api_key)

dims = prep.get_dims()
print(f"\nFeature dimensions: {dims}")

In [ ]:
# Pre-build samples ONCE with max lookback so all Optuna trials reuse
MAX_TECH_LOOKBACK = 128
MAX_NUMBARS_LOOKBACK = 24

print(f"Pre-building samples (tech={MAX_TECH_LOOKBACK}, nb={MAX_NUMBARS_LOOKBACK})...")
_all_samples = prep.build_samples(
    tech_lookback=MAX_TECH_LOOKBACK,
    numbars_lookback=MAX_NUMBARS_LOOKBACK,
    session_only=True,
    sample_session_start=SAMPLE_SESSION_START,
    sample_session_end=SAMPLE_SESSION_END,
    stride=SAMPLE_STRIDE,
)
assert _all_samples, "No valid samples produced. Check data availability."

# Date-based train/val split (80/20 chronological)
_all_dates = sorted(set(s["date"] for s in _all_samples))
_n_val = max(1, int(len(_all_dates) * 0.2))
VAL_CUTOFF = _all_dates[-_n_val]

TRAIN_SAMPLES = [s for s in _all_samples if s["date"] < VAL_CUTOFF]
VAL_SAMPLES = [s for s in _all_samples if s["date"] >= VAL_CUTOFF]
n_val_samples = len(VAL_SAMPLES)
del _all_samples

print(f"Cached samples: {len(TRAIN_SAMPLES)} train, {n_val_samples} val ")
print(f"Val cutoff: {VAL_CUTOFF}, {len(_all_dates)} total trading days")

F_TECH = dims['f_tech']
NB_CHANNELS = dims.get('nb_channels', 4)
NB_BINS = dims.get('nb_bins', 32)
RASTER_C = dims.get('raster_C', 4)
RASTER_BINS = dims.get('raster_bins', 64)
print(f"F_TECH={F_TECH}, NB={NB_CHANNELS}x{NB_BINS}, Raster={RASTER_C}x{RASTER_BINS}")

## 4. Model Wrapper + Train/Eval Functions

In [ ]:
from CTAFlow.models.deep_learning.multi_branch.macro_gat_model import MacroGATMamba
from CTAFlow.models.deep_learning.multi_branch.tft.c_mmtft import (
    PredictionToPosition,
    PTPLoss,
    returns_to_classes,
)
from CTAFlow.models.deep_learning.training.loss.clf import (
    ContinuousTradingLoss,
    SharpeScheduler,
)


def hybrid_selection_score(metrics: dict, n_samples: int) -> float:
    sharpe = float(metrics.get('sharpe', 0.0))
    sortino = float(metrics.get('sortino', 0.0))
    profit_factor = float(metrics.get('profit_factor', 1e-8))
    mean_strategy_ret = float(metrics.get('mean_strategy_ret', 0.0))

    total_return = mean_strategy_ret * float(n_samples)
    total_return_score = math.copysign(
        math.log1p(abs(total_return) * SELECTION_TOTAL_RETURN_SCALE),
        total_return,
    )
    log_pf = math.log(max(profit_factor, 1e-8))

    return (
        SELECTION_SHARPE_WEIGHT * sharpe
        + SELECTION_SORTINO_WEIGHT * sortino
        + SELECTION_PF_WEIGHT * log_pf
        + SELECTION_RETURN_WEIGHT * total_return_score
    )


class LogitPTP(nn.Module):
    """Learnable logit-to-position mapper.

    Takes raw 4-class logits from MacroGATMamba and converts them to
    continuous position in [-1, 1] via learnable anchors + temperature.
    Includes a small residual refiner that lets the network adjust
    the logit distribution before the anchor mapping.
    """

    N_CLASSES = 4

    def __init__(self, temperature: float = 1.5, dropout: float = 0.2):
        super().__init__()
        self.refiner = nn.Sequential(
            nn.Linear(self.N_CLASSES, 16),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(16, self.N_CLASSES),
        )
        self.ptp = PredictionToPosition(temperature=temperature)

    def forward(self, logits: torch.Tensor):
        """logits: (B, 4) -> (position (B,1), logits (B,4))"""
        refined = logits + self.refiner(logits)
        position, _ = self.ptp(refined)
        return position, refined

    def ptp_parameters(self):
        yield from self.ptp.parameters()
        yield from self.refiner.parameters()

    def get_last_stats(self):
        return self.ptp.get_last_stats()


class MacroGATPTP(nn.Module):
    """MacroGATMamba + LogitPTP wrapper.

    Runs the 3-branch model to produce 4-class logits, then maps
    logits -> continuous position via learnable anchor-based PTP.
    """

    def __init__(
        self,
        base_model: MacroGATMamba,
        ptp_temperature: float = 1.5,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.base_model = base_model
        self.head = LogitPTP(temperature=ptp_temperature, dropout=dropout)

    def forward(
        self,
        tech_features,
        numbars_recent,
        numbars_lens,
        raster_prev_day,
        macro_dict,
        return_tracker: bool = False,
    ):
        out = self.base_model(
            tech_features=tech_features,
            numbars_recent=numbars_recent,
            numbars_lens=numbars_lens,
            raster_prev_day=raster_prev_day,
            macro_dict=macro_dict,
            return_tracker=return_tracker,
        )
        if return_tracker:
            logits_raw, tracker = out
        else:
            logits_raw = out
            tracker = None

        position, logits = self.head(logits_raw)
        if tracker is not None:
            return position, logits, tracker
        return position, logits


def build_gat_ptp_param_groups(
    model: MacroGATPTP,
    base_lr: float,
    weight_decay: float = 0.0,
    ptp_lr_scale: float = 1.0,
) -> list:
    """Build optimizer groups: trunk (base_model) + ptp head."""
    ptp_params = list(model.head.ptp_parameters())
    ptp_ids = {id(p) for p in ptp_params}
    trunk_params = [p for p in model.parameters() if p.requires_grad and id(p) not in ptp_ids]

    groups = []
    if trunk_params:
        groups.append({"params": trunk_params, "lr": base_lr, "weight_decay": weight_decay, "group_name": "trunk"})
    if ptp_params:
        groups.append({"params": ptp_params, "lr": base_lr * ptp_lr_scale, "weight_decay": weight_decay, "group_name": "ptp_head"})
    return groups


def _make_loaders_from_cache(
    train_samples, val_samples, prep,
    batch_size: int = 64,
    tech_lookback: int = 64,
):
    """Build DataLoaders from pre-cached samples, truncating tech_features."""
    from torch.utils.data import DataLoader

    def _truncate(samples, tl):
        out = []
        for s in samples:
            s2 = dict(s)
            tf = s2['tech_features']
            if tf.shape[0] > tl:
                s2['tech_features'] = tf[-tl:]
            out.append(s2)
        return out

    tr = _truncate(train_samples, tech_lookback)
    vl = _truncate(val_samples, tech_lookback)

    raster_shape = prep._raster_shape or (12, 4, 64)
    macro_node_dims = {}
    for name in NODE_ORDER:
        dim_key = f"macro_{name}"
        if dim_key in dims:
            macro_node_dims[name] = dims[dim_key]

    train_ds = MacroGATDataset(tr, raster_shape=raster_shape,
                               macro_lookback=prep.macro_lookback,
                               macro_node_dims=macro_node_dims)
    val_ds = MacroGATDataset(vl, raster_shape=raster_shape,
                             macro_lookback=prep.macro_lookback,
                             macro_node_dims=macro_node_dims)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        collate_fn=macro_gat_collate_fn, num_workers=0,
        pin_memory=True, drop_last=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        collate_fn=macro_gat_collate_fn, num_workers=0,
        pin_memory=True,
    )
    return train_loader, val_loader

In [ ]:
def train_epoch_gat_ptp(
    model: MacroGATPTP,
    loader,
    ptp_loss: PTPLoss,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    max_norm: float = 1.0,
    scaler=None,
) -> Tuple[float, Dict[str, float]]:
    """Train one epoch with PTP multi-loss (CE + PnL)."""
    model.train()
    use_amp = scaler is not None
    total_loss = 0.0
    metric_accum = defaultdict(float)
    n_batches = 0
    prev_pos = None

    for batch in loader:
        intraday, macro_dict, targets, target_classes = unpack_macro_gat_batch(batch, device)
        targets = targets.float()

        optimizer.zero_grad()
        with torch.amp.autocast(device.type, enabled=use_amp):
            position, logits = model(
                tech_features=intraday['tech_features'],
                numbars_recent=intraday['numbars_recent'],
                numbars_lens=intraday['numbars_lens'],
                raster_prev_day=intraday['raster_prev_day'],
                macro_dict=macro_dict,
            )

        position = position.float()
        logits = logits.float()

        loss, ptp_metrics = ptp_loss(position, logits, targets, prev_position=prev_pos)

        if torch.isnan(loss) or torch.isinf(loss):
            continue

        if use_amp:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm)
            optimizer.step()

        total_loss += loss.item()
        for k, v in ptp_metrics.items():
            metric_accum[k] += v
        n_batches += 1
        prev_pos = position.detach()

    n = max(n_batches, 1)
    return total_loss / n, {k: v / n for k, v in metric_accum.items()}


@torch.no_grad()
def evaluate_gat_ptp(
    model: MacroGATPTP,
    loader,
    ptp_loss: PTPLoss,
    device: torch.device,
) -> Dict[str, float]:
    """Evaluate MacroGAT PTP model."""
    model.eval()
    all_positions, all_returns, all_logits = [], [], []
    metric_accum = defaultdict(float)
    total_loss = 0.0
    n_batches = 0

    for batch in loader:
        intraday, macro_dict, targets, target_classes = unpack_macro_gat_batch(batch, device)
        targets = targets.float()

        position, logits = model(
            tech_features=intraday['tech_features'],
            numbars_recent=intraday['numbars_recent'],
            numbars_lens=intraday['numbars_lens'],
            raster_prev_day=intraday['raster_prev_day'],
            macro_dict=macro_dict,
        )

        ptp_total, ptp_metrics = ptp_loss(position, logits, targets)
        total_loss += ptp_total.item()
        n_batches += 1
        for k, v in ptp_metrics.items():
            metric_accum[k] += v

        all_positions.append(position.view(-1).cpu())
        all_returns.append(targets.view(-1).cpu())
        all_logits.append(logits.cpu())

    positions = torch.cat(all_positions)
    returns = torch.cat(all_returns)
    all_logits_cat = torch.cat(all_logits)
    strategy_ret = positions * returns

    n = max(n_batches, 1)
    mean_r = strategy_ret.mean().item()
    std_r = strategy_ret.std().item() + 1e-8

    gross_profit = strategy_ret[strategy_ret > 0].sum().item()
    gross_loss = strategy_ret[strategy_ret < 0].abs().sum().item() + 1e-8
    cum_ret = strategy_ret.cumsum(dim=0)

    correct_dir = ((positions > 0) & (returns > 0)) | ((positions < 0) & (returns < 0))
    non_flat = positions.abs() > 0.05
    dir_acc = (correct_dir & non_flat).float().sum().item() / max(non_flat.float().sum().item(), 1)

    # Classification metrics
    class_labels = returns_to_classes(returns, outer=ptp_loss.outer_threshold)
    pred_cls = all_logits_cat.argmax(dim=-1)
    cls_acc = (pred_cls == class_labels).float().mean().item() * 100.0

    per_class = {}
    for c in range(4):
        mask = class_labels == c
        if mask.any():
            per_class[f'cls_{c}_acc'] = (pred_cls[mask] == c).float().mean().item() * 100.0
            per_class[f'cls_{c}_count'] = int(mask.sum().item())

    avg_exposure = positions.abs().mean().item()
    sharpe = mean_r / std_r
    sortino = mean_r / (strategy_ret.clamp(max=0.0).pow(2).mean().sqrt().item() + 1e-8)
    profit_factor = gross_profit / gross_loss

    # Selection score (same as reference)
    def _sel(sh, so, pf, exp):
        pf_c = min(pf, 5.0)
        exp_p = 1.0 if 0.15 <= exp <= 0.65 else max(0.3, 1.0 - 2.0 * abs(exp - 0.4))
        return (0.40 * sh + 0.25 * so + 0.20 * (pf_c - 1.0) + 0.15 * (exp_p - 0.5))

    metrics = {
        'loss': total_loss / n,
        'sharpe': sharpe,
        'sortino': sortino,
        'mean_strategy_ret': mean_r,
        'win_rate': (strategy_ret > 0).float().mean().item() * 100.0,
        'dir_accuracy': dir_acc * 100.0,
        'profit_factor': profit_factor,
        'max_drawdown': (cum_ret.cummax(dim=0)[0] - cum_ret).max().item(),
        'avg_exposure': avg_exposure,
        'avg_position': positions.mean().item(),
        'selection_score': _sel(sharpe, sortino, profit_factor, avg_exposure),
        'cls_accuracy': cls_acc,
        **per_class,
        'n_samples': len(positions),
    }
    metrics.update({k: v / n for k, v in metric_accum.items()})
    return metrics

## 5. Define Optuna Objective

In [ ]:
def objective(trial: optuna.Trial) -> float:
    # --- Architecture ---
    d_model = trial.suggest_categorical('d_model', [64, 128])
    macro_temporal_hidden = trial.suggest_categorical('macro_temporal_hidden', [32, 64])
    macro_num_heads = trial.suggest_categorical('macro_num_heads', [2, 4])
    n_mamba_layers = trial.suggest_int('n_mamba_layers', 1, 3)
    d_state = trial.suggest_categorical('d_state', [16, 32])
    d_conv = trial.suggest_categorical('d_conv', [2, 4])
    expand = trial.suggest_categorical('expand', [1, 2])
    dropout = trial.suggest_float('dropout', 0.1, 0.4)

    # --- PTP parameters ---
    ptp_temperature = trial.suggest_float('ptp_temperature', 1.0, 3.0)
    ce_weight = trial.suggest_float('ce_weight', 0.3, 2.0, log=True)
    pnl_weight = trial.suggest_float('pnl_weight', 0.3, 2.0, log=True)
    profit_scale = trial.suggest_float('profit_scale', 50.0, 200.0)
    outer_threshold = trial.suggest_float('outer_threshold', 0.8, 1.5)
    ptp_lr_scale = trial.suggest_float('ptp_lr_scale', 0.50, 1.25)

    # --- Training ---
    tech_lookback = trial.suggest_categorical('tech_lookback', [48, 64, 96])
    numbars_lookback = trial.suggest_categorical('numbars_lookback', [8, 12, 16])
    batch_size = trial.suggest_categorical('batch_size', [48, 64, 96])
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-3, 7e-3, log=True)
    max_norm = trial.suggest_float('max_norm', 0.5, 1.0)

    # --- ContinuousTradingLoss ---
    tc_cost = trial.suggest_float('tc_cost', 5e-5, 5e-4, log=True)
    init_direction_weight = trial.suggest_float('init_direction_weight', 0.5, 1.5)
    final_direction_weight = trial.suggest_float('final_direction_weight', 0.05, 0.3)
    init_reg_weight = trial.suggest_float('init_reg_weight', 0.1, 0.5)
    target_exposure = trial.suggest_float('target_exposure', 0.2, 0.5)
    downside_vol_weight = trial.suggest_float('downside_vol_weight', 0.02, 0.75, log=True)
    holding_weight = trial.suggest_float('holding_weight', 0.0, 0.5)
    exposure_asymmetry = trial.suggest_float('exposure_asymmetry', 1.0, 6.0)

    NUM_EPOCHS = 20
    WARMUP_EPOCHS = 5

    # --- Dataloaders ---
    try:
        train_loader, val_loader = _make_loaders_from_cache(
            TRAIN_SAMPLES, VAL_SAMPLES, prep,
            batch_size=batch_size,
            tech_lookback=tech_lookback,
        )
    except Exception as e:
        print(f"Dataloader failed: {e}")
        return -1e9

    # --- Model ---
    base_model = MacroGATMamba.from_prep_dims(
        dims,
        d_model=d_model,
        num_classes=4,
        macro_temporal_hidden=macro_temporal_hidden,
        macro_num_heads=macro_num_heads,
        n_mamba_layers=n_mamba_layers,
        d_state=d_state,
        d_conv=d_conv,
        expand=expand,
        dropout=dropout,
    )
    model = MacroGATPTP(
        base_model=base_model,
        ptp_temperature=ptp_temperature,
        dropout=dropout,
    ).to(device)

    # --- Loss ---
    trading_kwargs = dict(
        tc_cost=tc_cost,
        direction_weight=init_direction_weight,
        reg_weight=init_reg_weight,
        target_exposure=target_exposure,
        use_sortino=True,
        downside_vol_weight=downside_vol_weight,
        tc_in_sharpe=True,
        holding_weight=holding_weight,
        exposure_asymmetry=exposure_asymmetry,
    )
    loss_fn = PTPLoss(
        ce_weight=ce_weight,
        pnl_weight=pnl_weight,
        profit_scale=profit_scale,
        outer_threshold=outer_threshold,
        **trading_kwargs,
    ).to(device)

    # --- Optimizer with param groups ---
    optimizer = optim.AdamW(
        build_gat_ptp_param_groups(
            model, base_lr=learning_rate, weight_decay=weight_decay,
            ptp_lr_scale=ptp_lr_scale,
        )
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=learning_rate * 0.01)

    sharpe_sched = SharpeScheduler(
        warmup_epochs=WARMUP_EPOCHS,
        total_epochs=NUM_EPOCHS,
        initial_direction_weight=init_direction_weight,
        final_direction_weight=final_direction_weight,
        initial_reg_weight=init_reg_weight,
        final_reg_weight=0.05,
        initial_target_exposure=0.2,
        final_target_exposure=target_exposure,
        initial_holding_weight=0.0,
        final_holding_weight=holding_weight,
    )

    scaler = torch.amp.GradScaler() if (USE_AMP and device.type == 'cuda') else None

    best_score = -1e9
    patience_counter = 0
    prev_val_loss = None

    for epoch in range(NUM_EPOCHS):
        sharpe_sched.step(epoch, loss_fn.trading_loss)
        is_warmup = epoch < WARMUP_EPOCHS

        train_loss, train_metrics = train_epoch_gat_ptp(
            model, train_loader, loss_fn, optimizer, device,
            max_norm=max_norm, scaler=scaler,
        )
        val_metrics = evaluate_gat_ptp(
            model, val_loader, loss_fn, device,
        )

        scheduler.step()

        val_loss = val_metrics['loss']
        val_score = hybrid_selection_score(val_metrics, n_val_samples)

        # Early stopping guards
        if math.isnan(val_loss) or math.isinf(val_loss) or val_loss > 100.0:
            return best_score if best_score > -1e9 else -1e9
        if prev_val_loss is not None and epoch >= 3 and val_loss > abs(prev_val_loss) * 5.0:
            return best_score if best_score > -1e9 else -1e9
        prev_val_loss = val_loss

        print(
            f"  E{epoch+1:02d} | Score: {val_score:.4f} | Sharpe: {val_metrics['sharpe']:.4f} "
            f"| Sortino: {val_metrics['sortino']:.4f} | PF: {val_metrics['profit_factor']:.3f} "
            f"| MeanRet: {val_metrics['mean_strategy_ret']:.6f}"
            f"{' [warmup]' if is_warmup else ''}"
        )

        if is_warmup:
            continue

        if val_score > best_score:
            best_score = val_score
            patience_counter = 0
            trial.set_user_attr('final_selection_score', val_score)
            trial.set_user_attr('final_sharpe', val_metrics['sharpe'])
            trial.set_user_attr('final_sortino', val_metrics['sortino'])
            trial.set_user_attr('final_mean_strategy_ret', val_metrics['mean_strategy_ret'])
            trial.set_user_attr('final_win_rate', val_metrics['win_rate'])
            trial.set_user_attr('final_dir_acc', val_metrics['dir_accuracy'])
            trial.set_user_attr('final_pf', val_metrics['profit_factor'])
            trial.set_user_attr('final_exposure', val_metrics['avg_exposure'])
            trial.set_user_attr('final_loss', val_loss)
            trial.set_user_attr('final_downside_vol', val_metrics.get('downside_vol', 0.0))
            trial.set_user_attr('final_cls_acc', val_metrics.get('cls_accuracy', 0))
        else:
            patience_counter += 1

        trial.report(val_score, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
        if patience_counter >= 8:
            break

    del model, base_model, optimizer, loss_fn, train_loader, val_loader
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()
    return best_score

## 6. Run Optuna Optimization

In [ ]:
N_TRIALS = 30
prefix = f"macro_gat_{TICKER}_ptp"

study = optuna.create_study(
    direction="maximize",
    study_name=prefix,
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=7),
)
print(f"Study: {prefix}, {N_TRIALS} trials")

In [ ]:
study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True,
    gc_after_trial=True,
)

In [ ]:
# Best trial results
best_trial = study.best_trial
print(f"\nBest trial #{best_trial.number}:")
print(f"  Score: {best_trial.value:.4f}")
for k, v in sorted(best_trial.user_attrs.items()):
    print(f"  {k}: {v}")
print(f"\nBest params:")
best_params = best_trial.params
for k, v in sorted(best_params.items()):
    print(f"  {k}: {v}")

In [ ]:
# Save study artifacts
import joblib

joblib.dump(study, RESULTS_PATH / f"{prefix}_study.pkl")
with open(RESULTS_PATH / f"{prefix}_best_params.json", 'w') as f:
    json.dump(best_params, f, indent=2)
print(f"Study saved to {RESULTS_PATH}")

df_trials = study.trials_dataframe()
df_trials.to_csv(RESULTS_PATH / f"{prefix}_trials.csv", index=False)

## 7. Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

valid_trials = df_trials[df_trials['state'] == 'COMPLETE']

axes[0, 0].plot(valid_trials['number'], valid_trials['value'], 'o-', alpha=0.7)
axes[0, 0].axhline(best_trial.value, color='r', linestyle='--', label=f'Best: {best_trial.value:.4f}')
axes[0, 0].set_xlabel('Trial')
axes[0, 0].set_ylabel('Score')
axes[0, 0].set_title('Selection Score by Trial')
axes[0, 0].legend()

# Cumulative best
cum_best = valid_trials['value'].cummax()
axes[0, 1].plot(valid_trials['number'], cum_best, 'g-', linewidth=2)
axes[0, 1].set_xlabel('Trial')
axes[0, 1].set_ylabel('Best Score')
axes[0, 1].set_title('Cumulative Best')

# d_model distribution
if 'params_d_model' in valid_trials.columns:
    valid_trials.boxplot(column='value', by='params_d_model', ax=axes[1, 0])
    axes[1, 0].set_title('Score by d_model')
    axes[1, 0].set_xlabel('d_model')

# Learning rate vs Score
if 'params_learning_rate' in valid_trials.columns:
    axes[1, 1].scatter(valid_trials['params_learning_rate'], valid_trials['value'], alpha=0.6)
    axes[1, 1].set_xscale('log')
    axes[1, 1].set_xlabel('Learning Rate')
    axes[1, 1].set_ylabel('Score')
    axes[1, 1].set_title('LR vs Score')

plt.suptitle(f'MacroGAT {TICKER} Optuna Results', fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{prefix}_optuna_overview.png", dpi=150)
plt.show()

## 8. Train Final Model with Best Parameters

In [ ]:
best = best_params
print("Training final model with best parameters:")
for k, v in sorted(best.items()):
    print(f"  {k}: {v}")

In [ ]:
# Build final dataloaders + model
tech_lookback = best['tech_lookback']
numbars_lookback = best.get('numbars_lookback', 8)

train_loader, val_loader = _make_loaders_from_cache(
    TRAIN_SAMPLES, VAL_SAMPLES, prep,
    batch_size=best['batch_size'],
    tech_lookback=tech_lookback,
)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

base_model = MacroGATMamba.from_prep_dims(
    dims,
    d_model=best['d_model'],
    num_classes=4,
    macro_temporal_hidden=best['macro_temporal_hidden'],
    macro_num_heads=best['macro_num_heads'],
    n_mamba_layers=best['n_mamba_layers'],
    d_state=best.get('d_state', 16),
    d_conv=best.get('d_conv', 4),
    expand=best.get('expand', 2),
    dropout=best['dropout'],
)

final_model = MacroGATPTP(
    base_model=base_model,
    ptp_temperature=best.get('ptp_temperature', 1.5),
    dropout=best['dropout'],
).to(device)

print(f"Model parameters: {sum(p.numel() for p in final_model.parameters()):,}")

In [ ]:
NUM_EPOCHS = 30
WARMUP_EPOCHS = 5

trading_kwargs = dict(
    tc_cost=best['tc_cost'],
    direction_weight=best['init_direction_weight'],
    reg_weight=best['init_reg_weight'],
    target_exposure=best['target_exposure'],
    use_sortino=True,
    downside_vol_weight=best.get('downside_vol_weight', 0.1),
    tc_in_sharpe=True,
    holding_weight=best.get('holding_weight', 0.0),
    exposure_asymmetry=best.get('exposure_asymmetry', 3.0),
)

loss_fn = PTPLoss(
    ce_weight=best.get('ce_weight', 1.0),
    pnl_weight=best.get('pnl_weight', 1.0),
    profit_scale=best.get('profit_scale', 100.0),
    outer_threshold=best.get('outer_threshold', 1.0),
    **trading_kwargs,
).to(device)

optimizer = optim.AdamW(
    build_gat_ptp_param_groups(
        final_model,
        base_lr=best['learning_rate'],
        weight_decay=best['weight_decay'],
        ptp_lr_scale=best.get('ptp_lr_scale', 1.0),
    )
)

lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=best['learning_rate'] * 0.01,
)

sharpe_sched = SharpeScheduler(
    warmup_epochs=WARMUP_EPOCHS,
    total_epochs=NUM_EPOCHS,
    initial_direction_weight=best['init_direction_weight'],
    final_direction_weight=best['final_direction_weight'],
    initial_reg_weight=best['init_reg_weight'],
    final_reg_weight=0.05,
    initial_target_exposure=0.2,
    final_target_exposure=best['target_exposure'],
    initial_holding_weight=0.0,
    final_holding_weight=best.get('holding_weight', 0.0),
)

history = {
    'train_loss': [], 'val_loss': [],
    'val_sharpe': [], 'val_sortino': [],
    'val_score': [],
    'val_win_rate': [], 'val_dir_acc': [],
    'val_pf': [], 'val_exposure': [],
    'val_max_dd': [], 'val_downside_vol': [], 'lr': [],
    'direction_weight': [], 'reg_weight': [],
    'val_cls_acc': [],
    'val_ce_loss': [],
    'train_ce_loss': [],
    'train_trading_loss': [],
    'val_trading_loss': [],
}

best_score = -1e9
best_state = None

scaler = torch.amp.GradScaler() if (USE_AMP and device.type == 'cuda') else None
if scaler:
    print('Mixed precision (AMP) enabled')

print(f"Training for {NUM_EPOCHS} epochs (MacroGATMamba + PTP)")
print(f"Loss: PTPLoss (CE + downside-aware PnL) + SharpeScheduler (warmup={WARMUP_EPOCHS})")
print("=" * 100)

for epoch in range(NUM_EPOCHS):
    sharpe_sched.step(epoch, loss_fn.trading_loss)

    train_loss, train_metrics = train_epoch_gat_ptp(
        final_model, train_loader, loss_fn, optimizer, device,
        max_norm=best['max_norm'], scaler=scaler,
    )
    val_metrics = evaluate_gat_ptp(
        final_model, val_loader, loss_fn, device,
    )

    lr_scheduler.step()

    val_score = hybrid_selection_score(val_metrics, n_val_samples)

    # History tracking
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_metrics['loss'])
    history['val_sharpe'].append(val_metrics['sharpe'])
    history['val_sortino'].append(val_metrics['sortino'])
    history['val_score'].append(val_score)
    history['val_win_rate'].append(val_metrics['win_rate'])
    history['val_dir_acc'].append(val_metrics['dir_accuracy'])
    history['val_pf'].append(val_metrics['profit_factor'])
    history['val_exposure'].append(val_metrics['avg_exposure'])
    history['val_max_dd'].append(val_metrics['max_drawdown'])
    history['val_downside_vol'].append(val_metrics.get('downside_vol', 0.0))
    history['val_cls_acc'].append(val_metrics.get('cls_accuracy', 0.0))
    history['val_ce_loss'].append(val_metrics.get('ce_loss', 0.0))
    history['train_ce_loss'].append(train_metrics.get('ce_loss', 0.0))
    history['train_trading_loss'].append(train_metrics.get('trading_loss', 0.0))
    history['val_trading_loss'].append(val_metrics.get('trading_loss', 0.0))
    history['lr'].append(optimizer.param_groups[0]['lr'])
    history['direction_weight'].append(loss_fn.trading_loss.direction_weight)
    history['reg_weight'].append(loss_fn.trading_loss.reg_weight)

    is_warmup = epoch < WARMUP_EPOCHS

    print(
        f"  E{epoch+1:02d} | Score: {val_score:.4f} | Sharpe: {val_metrics['sharpe']:.4f} "
        f"| Sortino: {val_metrics['sortino']:.4f} | PF: {val_metrics['profit_factor']:.3f} "
        f"| MeanRet: {val_metrics['mean_strategy_ret']:.6f}"
        f"{' [warmup]' if is_warmup else ''}"
    )

    if not is_warmup and val_score > best_score:
        best_score = val_score
        best_state = deepcopy(final_model.state_dict())

In [ ]:
# Load best model state
if best_state:
    final_model.load_state_dict(best_state)
    print(f"Loaded best model (Score: {best_score:.4f})")

# Final evaluation
final_metrics = evaluate_gat_ptp(final_model, val_loader, loss_fn, device)
print("\nFinal validation metrics:")
for k, v in sorted(final_metrics.items()):
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

In [ ]:
# Plot training history
fig, axes = plt.subplots(3, 3, figsize=(18, 15))

# Loss
axes[0, 0].plot(history['train_loss'], label='Train')
axes[0, 0].plot(history['val_loss'], label='Val')
axes[0, 0].set_title('Total Loss')
axes[0, 0].legend()

# Sharpe
axes[0, 1].plot(history['val_sharpe'], 'g-', linewidth=2)
axes[0, 1].set_title('Val Sharpe')

# Win Rate
axes[0, 2].plot(history['val_win_rate'])
axes[0, 2].set_title('Val Win Rate %')

# Exposure
axes[1, 0].plot(history['val_exposure'])
axes[1, 0].set_title('Val Avg Exposure')

# Direction Accuracy
axes[1, 1].plot(history['val_dir_acc'])
axes[1, 1].set_title('Val Dir Accuracy %')

# Classification Accuracy
axes[1, 2].plot(history['val_cls_acc'])
axes[1, 2].set_title('Val Classification Accuracy %')

# CE Loss
axes[2, 0].plot(history['train_ce_loss'], label='Train')
axes[2, 0].plot(history['val_ce_loss'], label='Val')
axes[2, 0].set_title('CE Loss')
axes[2, 0].legend()

# Trading Loss
axes[2, 1].plot(history['train_trading_loss'], label='Train')
axes[2, 1].plot(history['val_trading_loss'], label='Val')
axes[2, 1].set_title('Trading Loss')
axes[2, 1].legend()

# LR schedule
axes[2, 2].plot(history['lr'], label='LR')
axes[2, 2].set_title('Learning Rate')
axes[2, 2].set_yscale('log')
axes[2, 2].legend()

plt.suptitle(f'MacroGAT {TICKER} Training History', fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{prefix}_training_history.png", dpi=150)
plt.show()

## 9. Model Diagnostics -- GAT Attention Visualization

In [ ]:
# GAT attention weights visualization
final_model.eval()

# Get a val batch for diagnostics
val_batch = next(iter(val_loader))
intraday, macro_dict, targets, target_classes = unpack_macro_gat_batch(val_batch, device)

with torch.no_grad():
    position, logits, tracker = final_model(
        tech_features=intraday['tech_features'],
        numbars_recent=intraday['numbars_recent'],
        numbars_lens=intraday['numbars_lens'],
        raster_prev_day=intraday['raster_prev_day'],
        macro_dict=macro_dict,
        return_tracker=True,
    )

# Plot GAT attention heatmap
macro_attn = tracker['macro_attn'].cpu().numpy()  # (B, num_nodes, num_nodes)
avg_attn = macro_attn.mean(axis=0)  # Average over batch

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
im = ax.imshow(avg_attn, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(NODE_ORDER)))
ax.set_yticks(range(len(NODE_ORDER)))
ax.set_xticklabels(NODE_ORDER, rotation=45, ha='right')
ax.set_yticklabels(NODE_ORDER)
ax.set_title('MacroGAT Attention Weights (Batch Average)')
plt.colorbar(im)
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{prefix}_gat_attention.png", dpi=150)
plt.show()

# Print Gold row attention
gold_idx = NODE_ORDER.index('Gold')
print("\nGold node attention to other nodes:")
for i, name in enumerate(NODE_ORDER):
    print(f"  Gold -> {name}: {avg_attn[gold_idx, i]:.4f}")

In [ ]:
# Position distribution analysis
final_model.eval()
all_pos, all_ret, all_cls = [], [], []

with torch.no_grad():
    for batch in val_loader:
        intraday, macro_dict, targets, target_classes = unpack_macro_gat_batch(batch, device)
        position, logits = final_model(
            tech_features=intraday['tech_features'],
            numbars_recent=intraday['numbars_recent'],
            numbars_lens=intraday['numbars_lens'],
            raster_prev_day=intraday['raster_prev_day'],
            macro_dict=macro_dict,
        )
        all_pos.append(position.view(-1).cpu())
        all_ret.append(targets.cpu())
        all_cls.append(logits.argmax(dim=-1).cpu())

positions = torch.cat(all_pos).numpy()
returns = torch.cat(all_ret).numpy()
pred_classes = torch.cat(all_cls).numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].hist(positions, bins=50, alpha=0.7, edgecolor='black')
axes[0].set_title('Position Distribution')
axes[0].set_xlabel('Position')
axes[0].axvline(0, color='r', linestyle='--')

axes[1].scatter(returns, positions, alpha=0.1, s=2)
axes[1].set_xlabel('1h Forward Return')
axes[1].set_ylabel('Position')
axes[1].set_title('Position vs Return')
axes[1].axhline(0, color='r', linestyle='--', alpha=0.5)
axes[1].axvline(0, color='r', linestyle='--', alpha=0.5)

class_names = ['Strong-', 'Weak-', 'Weak+', 'Strong+']
class_counts = [np.sum(pred_classes == c) for c in range(4)]
axes[2].bar(class_names, class_counts, color=['red', 'salmon', 'lightgreen', 'green'])
axes[2].set_title('Predicted Class Distribution')

plt.suptitle(f'MacroGAT {TICKER} Position Analysis', fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{prefix}_position_analysis.png", dpi=150)
plt.show()

## 10. Save Final Model

In [ ]:
model_path = RESULTS_PATH / f"{prefix}_best_model.pth"

save_dict = {
    'model_state_dict': final_model.state_dict(),
    'best_params': best_params,
    'dims': dims,
    'history': history,
    'final_metrics': final_metrics,
    'config': {
        'ticker': TICKER,
        'target_horizon_minutes': TARGET_HORIZON_MINUTES,
        'bar_minutes': BAR_MINUTES,
        'session_start': SAMPLE_SESSION_START,
        'session_end': SAMPLE_SESSION_END,
        'stride': SAMPLE_STRIDE,
        'macro_lookback': MACRO_LOOKBACK,
        'quartile_window': QUARTILE_WINDOW,
        'val_cutoff': str(VAL_CUTOFF),
    },
}
torch.save(save_dict, model_path)
print(f"Model saved to {model_path}")
print(f"Best Score: {best_score:.4f}")

## 11. Validation Backtest

In [ ]:
# Validation Backtest
final_model.eval()
all_pos, all_ret, all_dates = [], [], []

with torch.no_grad():
    for batch in val_loader:
        intraday, macro_dict, targets, target_classes = unpack_macro_gat_batch(batch, device)
        position, logits = final_model(
            tech_features=intraday['tech_features'],
            numbars_recent=intraday['numbars_recent'],
            numbars_lens=intraday['numbars_lens'],
            raster_prev_day=intraday['raster_prev_day'],
            macro_dict=macro_dict,
        )
        all_pos.append(position.view(-1).cpu().numpy())
        all_ret.append(targets.cpu().numpy())

positions = np.concatenate(all_pos)
returns = np.concatenate(all_ret)
strategy_returns = positions * returns

cum_strategy = np.cumsum(strategy_returns)
cum_bh = np.cumsum(returns)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(cum_strategy, label='Strategy (MacroGAT PTP)', linewidth=1.5)
axes[0].plot(cum_bh, label='Buy & Hold', linewidth=1, alpha=0.6)
axes[0].set_title(f'MacroGAT {TICKER} Validation Backtest')
axes[0].set_ylabel('Cumulative Return')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].fill_between(range(len(positions)), positions, alpha=0.5, color='steelblue')
axes[1].set_ylabel('Position')
axes[1].set_xlabel('Sample')
axes[1].axhline(0, color='black', linestyle='--', alpha=0.3)
axes[1].set_ylim(-1.1, 1.1)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{prefix}_backtest.png", dpi=150)
plt.show()

# Summary stats
sr = strategy_returns
print(f"\nBacktest Summary:")
print(f"  Total Samples: {len(sr)}")
print(f"  Cumulative Return: {sr.sum():.4f}")
print(f"  Mean Return: {sr.mean():.6f}")
print(f"  Sharpe: {sr.mean() / (sr.std() + 1e-8):.4f}")
print(f"  Win Rate: {(sr > 0).mean() * 100:.1f}%")
print(f"  Avg Exposure: {np.abs(positions).mean():.3f}")
print(f"  Max Drawdown: {(np.maximum.accumulate(np.cumsum(sr)) - np.cumsum(sr)).max():.4f}")